In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
# import ipywidgets as widgets
from IPython.display import display

# 載入你的模型推論函數與類別名稱
from inference import predict, CLASS_NAMES

# ==========================================
# 1. 設定資料夾路徑 (請替換成你的實際路徑)
# ==========================================
BASE_DIR = Path("D:/hagrid_project/dataset_v1_processed_detected")
CROP_DIR = BASE_DIR / "crops"
LMK_DIR = BASE_DIR / "landmarks"

# 抓取所有處理好的圖片
all_images = list(CROP_DIR.glob("*.jpg"))
if not all_images:
    print("找不到圖片，請檢查路徑設定！")

# ==========================================
# 2. 建立視覺化與預測的核心函數
# ==========================================
def show_prediction(image_index):
    if not all_images:
        return
        
    # 取得目前的圖片路徑與座標路徑
    img_path = all_images[image_index]
    base_name = img_path.stem
    lmk_path = LMK_DIR / f"{base_name}.npy"
    
    if not lmk_path.exists():
        print(f"找不到座標檔: {lmk_path.name}")
        return
        
    # 讀取圖片與轉換顏色 (BGR -> RGB)
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # 讀取座標
    landmarks = np.load(str(lmk_path))
    
    # 讓你的模型進行預測
    try:
        pred_class_id = predict(img_rgb, landmarks)
        pred_name = CLASS_NAMES.get(pred_class_id, "N/A (未知)")
    except Exception as e:
        pred_name = f"預測錯誤: {e}"
        pred_class_id = -1
    
    # ==========================================
    # 3. 畫圖呈現
    # ==========================================
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img_rgb)
    
    # 將 21 個關節點畫在圖片上 (紅點)
    h, w, _ = img_rgb.shape
    for (x, y) in landmarks:
        # 如果你的座標是 0~1 的相對座標，需要乘上圖片長寬；若是絕對座標則不用
        if x <= 1.0 and y <= 1.0:
            ax.plot(x * w, y * h, 'ro', markersize=5)
        else:
            ax.plot(x, y, 'ro', markersize=5)
            
    # 設定標題 (顯示檔名與預測結果)
    title_color = 'green' if pred_class_id != 0 else 'red' # 如果判斷為 N/A 顯示紅色
    ax.set_title(f"檔案: {img_path.name}\n模型預測: {pred_name} (Class {pred_class_id})", 
                 fontsize=14, fontweight='bold', color=title_color)
    ax.axis('off')
    plt.show()

# ==========================================
# 4. 綁定互動式拉桿 (Slider)
# ==========================================
print(f"成功載入 {len(all_images)} 張圖片！請拖動下方拉桿切換圖片：")

# 建立一個拉桿，範圍從 0 到圖片總數
slider = widgets.IntSlider(
    value=0, 
    min=0, 
    max=max(0, len(all_images)-1), 
    step=1, 
    description='圖片編號:',
    layout=widgets.Layout(width='500px')
)

# 使用 interact 自動將拉桿與畫圖函數綁定
widgets.interact(show_prediction, image_index=slider);

成功載入 9988 張圖片！請拖動下方拉桿切換圖片：


NameError: name 'widgets' is not defined